# 兩雀掃描牌 — 訓練台灣牌辨識模型 (Jon Chan 資料集)

把 Jon Chan 的公開台灣牌資料集(3505 張，已標註好，42 類含 8 花牌)訓成 YOLOv8n，匯出成端上瀏覽器可跑的 ONNX。

**你只要做三件事：**
1. 右上角 `連線` → `變更執行階段類型` → 選 **T4 GPU**（免費）。
2. 在 Cell 2 貼上你的免費 Roboflow API key（版本號會自動抓最新，不用填）。
3. `執行階段` → `全部執行`，等 1–2 小時，最後會自動下載 `best.onnx`。

把下載到的 `best.onnx` 和 **Cell 7 印出的 NAMES 清單** 丟回給我，我接上就上線。

In [ ]:
# Cell 1 — 安裝套件
!pip install -q ultralytics roboflow onnx onnxslim pyyaml
import torch
print('CUDA 可用:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '無（記得切 T4 GPU！）')

In [ ]:
# Cell 2 — 下載 Jon Chan 台灣牌資料集（自動抓最新版，你只要填 key）
#   API key：登入 https://roboflow.com（免費）→ 右上頭像 → Settings → API Keys → 複製 Private API Key
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'PASTE_YOUR_FREE_KEY_HERE'   # <-- 換成你的（唯一必填）

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('jon-chan-gnsoa').project('mahjong-baq4s')
# 自動挑最新版本號（83 版裡挑最大的）
latest = max(int(v.version.split('/')[-1]) for v in project.versions())
print('使用版本:', latest)
dataset = project.version(latest).download('yolov8')
print('資料集下載到:', dataset.location)

In [ ]:
# Cell 3 — 印出類別清單（把這段輸出整個複製回傳給 Claude！）
import yaml
d = yaml.safe_load(open(dataset.location + '/data.yaml'))
names = d['names'] if isinstance(d['names'], list) else [d['names'][i] for i in sorted(d['names'])]
print('類別數:', len(names))
print('NAMES =', names)

In [ ]:
# Cell 4 — 訓練（yolov8n = 最小，適合手機端上；T4 約 40–70 分）
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=dataset.location + '/data.yaml',
            epochs=60, imgsz=640, batch=16, patience=15, name='tw_mahjong')

In [ ]:
# Cell 5 — 驗證準度（看 mAP50，>0.9 就很夠用）
best = 'runs/detect/tw_mahjong/weights/best.pt'
metrics = YOLO(best).val()
print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)

In [ ]:
# Cell 6 — 匯出 ONNX（opset 12，onnxruntime-web 相容；不含內建 NMS，符合 detect.js 的解碼器）
onnx_path = YOLO(best).export(format='onnx', imgsz=640, opset=12, simplify=True)
print('ONNX:', onnx_path)
import os; print('大小:', round(os.path.getsize(onnx_path)/1e6, 1), 'MB')

In [ ]:
# Cell 7 — 下載 ONNX 到你電腦（連同再印一次 NAMES）
print('NAMES =', names)
from google.colab import files
files.download(onnx_path)